<a href="https://colab.research.google.com/github/Epsi-Tech/Development-of-AI-Based-Knowledge-Retrieval-Platform-with-Query-Resolution-System/blob/main/sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Install the required SQL extension
!pip install ipython-sql --quiet

# 2. Load the SQL library into Colab's memory
%load_ext sql

# 3. Create and connect to a fresh, clean database
%sql sqlite://



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 55.1 MB/s eta 0:00:00


In [2]:
%load_ext sql
%config SqlMagic.style = 'PLAIN'  # Forces a safe, simple table style to bypass the bug
# %config SqlMagic.displaystyle = 'default' # Use this instead if you are using jupysql


The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [3]:
%config SqlMagic.style = '_DEPRECATED_DEFAULT'


In [5]:
#1.
# This SQL code creates a database table for employee records, inserts five sample rows, and retrieves all the data to verify the setup.
# id: A unique identifier for each employee. PRIMARY KEY ensures no two employees have the same ID and speeds up searches.
# name, department, hire_date: Columns that hold text data.
# salary: A column that holds whole numbers (integers).


%%sql
CREATE TABLE employees (
    id INTEGER PRIMARY KEY,
    name TEXT,
    department TEXT,
    salary INTEGER,
    hire_date TEXT
);

INSERT INTO employees VALUES (1, '  alice smith  ', 'Engineering', 85000, '2023-03-15');
INSERT INTO employees VALUES (2, 'bob jones', 'Marketing', 62000, '2022-07-22');
INSERT INTO employees VALUES (3, 'Charlie Brown', 'Engineering', 95000, '2021-11-05');
INSERT INTO employees VALUES (4, 'Diana prince', 'HR', 55000, '2024-01-10');
INSERT INTO employees VALUES (5, 'evan wright', 'Marketing', 70000, '2023-08-19');

-- 5. Run a test query to confirm it works
SELECT * FROM employees;


 * sqlite://
Done.
1 rows affected.
1 rows affected.
1 rows affected.
1 rows affected.
1 rows affected.
Done.


id,name,department,salary,hire_date
1,alice smith,Engineering,85000,2023-03-15
2,bob jones,Marketing,62000,2022-07-22
3,Charlie Brown,Engineering,95000,2021-11-05
4,Diana prince,HR,55000,2024-01-10
5,evan wright,Marketing,70000,2023-08-19


In [6]:

%%sql
SELECT * FROM employees;


 * sqlite://
Done.


id,name,department,salary,hire_date
1,alice smith,Engineering,85000,2023-03-15
2,bob jones,Marketing,62000,2022-07-22
3,Charlie Brown,Engineering,95000,2021-11-05
4,Diana prince,HR,55000,2024-01-10
5,evan wright,Marketing,70000,2023-08-19


In [7]:
# 2.Window Functions (RANK) -Rank employees by their salary within each department.
# PARTITION BY department: This chops your data into distinct groups (or "windows") based on the department. The ranking calculations start completely fresh for each department (e.g., Engineering gets its own 1, 2, 3... and Marketing gets its own 1, 2, 3...).
# ORDER BY salary DESC: Within each department pool, it sorts the employees from the highest salary to the lowest salary before assigning numbers.
# ROW_NUMBER(): Assigns a sequential, unique integer to each row starting at 1. It never skips a number and never duplicates a number, even if two employees earn the exact same salary.
# RANK(): Assigns a rank based on the value. If two employees tie for the highest salary, both get a rank of 1. The next person in line would skip a number and receive a rank of 3.

%%sql
SELECT
    department,
    name,
    salary,
    ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary DESC) AS row_num,
    RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS salary_rank
FROM employees;


 * sqlite://
Done.


department,name,salary,row_num,salary_rank
Engineering,Charlie Brown,95000,1,1
Engineering,alice smith,85000,2,2
HR,Diana prince,55000,1,1
Marketing,evan wright,70000,1,1
Marketing,bob jones,62000,2,2


In [8]:
# 3.String Manipulation- It uses standard SQL string functions to clean up employee names, generate abbreviations, count characters, and combine fields into a readable format.
# TRIM(name) AS cleaned_name: The TRIM() function removes any accidental spaces from the beginning and end of the names. For example, ' alice smith ' is cleaned up to become 'alice smith'.
# UPPER(SUBSTR(TRIM(name), 1, 3)) AS short_code: This extracts the first 3 letters of the cleaned name using SUBSTR() and converts them to capital letters using UPPER(). 'alice smith' becomes 'ALI'.
# LENGTH(TRIM(name)) AS character_count: This counts the total number of characters in the name after removing the outer spaces. Space between first and last names counts as a character, so 'alice smith' returns 11.
# TRIM(name) || ' (' || department || ')' AS employee_profile: The double pipes (||) act as glue to link text together. This creates a formatted profile label by putting the department name in parentheses next to the name. For example: 'alice smith (Engineering)'.


%%sql
SELECT
    name AS original_name,
    TRIM(name) AS cleaned_name,
    UPPER(SUBSTR(TRIM(name), 1, 3)) AS short_code,
    LENGTH(TRIM(name)) AS character_count,
    TRIM(name) || ' (' || department || ')' AS employee_profile
FROM employees;


 * sqlite://
Done.


original_name,cleaned_name,short_code,character_count,employee_profile
alice smith,alice smith,ALI,11,alice smith (Engineering)
bob jones,bob jones,BOB,9,bob jones (Marketing)
Charlie Brown,Charlie Brown,CHA,13,Charlie Brown (Engineering)
Diana prince,Diana prince,DIA,12,Diana prince (HR)
evan wright,evan wright,EVA,11,evan wright (Marketing)


In [9]:
# 4.Date and Time Functions-This query uses Date and Time Functions (specifically SQLite's STRFTIME function) along with string cleaning. It breaks down a full date into its individual year, month, and day components.
# TRIM(name) AS employee: Just like in the previous query, this cleans up any accidental leading or trailing spaces from the name field.
# STRFTIME(...): This is the "String Format Time" function used in SQLite. It reads a date string (formatted as YYYY-MM-DD) and extracts only the part you ask for based on the percentage % placeholder:
# '%Y': Extracts the 4-digit year.
# '%m': Extracts the 2-digit month (01 through 12).
# '%d': Extracts the 2-digit day of the month (01 through 31).

%%sql
SELECT
    TRIM(name) AS employee,
    hire_date,
    STRFTIME('%Y', hire_date) AS hire_year,
    STRFTIME('%m', hire_date) AS hire_month,
    STRFTIME('%d', hire_date) AS hire_day
FROM employees;


 * sqlite://
Done.


employee,hire_date,hire_year,hire_month,hire_day
alice smith,2023-03-15,2023,03,15
bob jones,2022-07-22,2022,07,22
Charlie Brown,2021-11-05,2021,11,05
Diana prince,2024-01-10,2024,01,10
evan wright,2023-08-19,2023,08,19


In [10]:
# 5.Mathematical and Arithmetic Functions- It calculates how much each employee's salary deviates from a target or benchmark figure of 73,400
# salary - 73400 AS raw_difference: This finds the basic difference. If an employee makes more than $73,400, this number is positive. If they make less, this number is negative.
# ABS(salary - 73400) AS absolute_difference: The ABS() function returns the absolute value. It strips away the negative signs, showing you purely how far away the salary is from the target, regardless of whether it is higher or lower.
# ROUND((... / 73400.0) * 100, 1) || '%': This calculates what percentage the absolute difference is relative to the $73,400 benchmark:
# Dividing by 73400.0 ensures the database calculates a decimal fraction instead of truncating it to a whole integer.
# ROUND(..., 1) limits the result to 1 decimal place.
# || '%' glues a percentage sign to the end for display.


%%sql
SELECT
    TRIM(name) AS employee,
    salary,
    salary - 73400 AS raw_difference,
    ABS(salary - 73400) AS absolute_difference,
    ROUND((ABS(salary - 73400) / 73400.0) * 100, 1) || '%' AS percent_deviation
FROM employees;


 * sqlite://
Done.


employee,salary,raw_difference,absolute_difference,percent_deviation
alice smith,85000,11600,11600,15.8%
bob jones,62000,-11400,11400,15.5%
Charlie Brown,95000,21600,21600,29.4%
Diana prince,55000,-18400,18400,25.1%
evan wright,70000,-3400,3400,4.6%


In [11]:
#  6.Filtering and Sorting - This query performs data Filtering and Sorting using a basic conditional clause and an ordering clause. Rather than manipulating the columns themselves, it restricts which rows are returned and dictates their exact order.
#  Instead of pulling every column from the table, it only fetches three specific fields: the employee's name, their department, and their hire_date
#  It tells the database engine to search inside the table named employees
#  WHERE hire_date > '2023-01-01' WHERE clause ensures that the query only returns records for employees whose start date is strictly after January 1, 2023. Any employee hired on or before this date is excluded from the final results
# ORDER BY hire_date DESC; By default, SQL sorts data in ascending order (ASC). The DESC keyword explicitly forces a descending order. For dates, sorting in descending order means the most recent (newest) hire dates will appear at the very top of the list, moving backward in time as you scroll down.



%%sql
SELECT name, department, hire_date
FROM employees
WHERE hire_date > '2023-01-01'
ORDER BY hire_date DESC;



 * sqlite://
Done.


name,department,hire_date
Diana prince,HR,2024-01-10
evan wright,Marketing,2023-08-19
alice smith,Engineering,2023-03-15


In [12]:
# 7.Filtering - This query performs data Filtering using multi-conditional logical operators.
# This query finds all information for high-earning software professionals. Specifically, it looks for employees who work in the Engineering department and make a salary strictly greater than 70,000.
# WHERE salary > 70000 AND department = 'Engineering';
# The WHERE clause uses the AND operator, meaning a row must meet both conditions to be included in the final results:
# salary > 70000: The employee's salary must be more than 70,000. Anyone making exactly 70,000 or less is filtered out.
# department = 'Engineering': The employee must belong specifically to the 'Engineering' team.


%%sql
SELECT *
FROM employees
WHERE salary > 70000
  AND department = 'Engineering';


 * sqlite://
Done.


id,name,department,salary,hire_date
1,alice smith,Engineering,85000,2023-03-15
3,Charlie Brown,Engineering,95000,2021-11-05


In [13]:
# 8.Aggregate Functions - This is an SQL query, not a built-in SQL function. More specifically, it is a data analysis query that uses several SQL Aggregate Functions alongside clauses to group and sort data.
# department: Displays the name of the group being analyzed.
# COUNT(*) AS total_employees: Counts every single row within that department bucket. This gives you the total headcount for each team.
# AVG(salary) AS average_salary: Calculates the mathematical mean of all salaries in that specific department.
# SUM(salary) AS total_payroll: Adds up every single salary in that department to calculate the total money spent on salaries for that team.
# AS [alias]: The AS keyword gives these calculated columns clean, descriptive names in your final report.
# ORDER BY total_payroll DESC; the calculated total_payroll for each department and sorts them in descending order.

%%sql
SELECT
    department,
    COUNT(*) AS total_employees,
    AVG(salary) AS average_salary,
    SUM(salary) AS total_payroll
FROM employees
GROUP BY department
ORDER BY total_payroll DESC;


 * sqlite://
Done.


department,total_employees,average_salary,total_payroll
Engineering,2,90000.0,180000
Marketing,2,66000.0,132000
HR,1,55000.0,55000


In [14]:
# 9.Self-Join - This is an SQL query that performs a Self-Join. A self-join is not a distinct built-in function, but rather a technique where a table is joined with itself (in this case, matching the employees table alias e1 to the same employees table alias e2).
# This query finds all unique pairs of coworkers within the same department. It generates a list of every possible combination of two employees who share a team, making sure it doesn't match a person with themselves or list the same pair twice (e.g., if it pairs Alice and Bob, it won't separately list Bob and Alice).
# The Self-Join (FROM employees e1 INNER JOIN employees e2)To compare rows within the exact same table, SQL allows you to open two virtual copies of it simultaneously. The query uses aliases (e1 and e2) to give these identical copies distinct names so the database knows which one you are referencing.
# (ON e1.department = e2.department) The INNER JOIN matches rows between copy e1 and copy e2. This first condition ensures that an engineer from e1 is only paired up with other engineers from e2.
# (AND e1.id < e2.id) This is the cleverest part of the query. Without this line, the system would create messy, redundant data. The less-than operator (<) accomplishes two critical things:

%%sql
SELECT
    e1.name AS employee_1,
    e2.name AS employee_2,
    e1.department
FROM employees e1
INNER JOIN employees e2
    ON e1.department = e2.department
    AND e1.id < e2.id; -- Prevents pairing an employee with themselves or duplicating pairs


 * sqlite://
Done.


employee_1,employee_2,department
alice smith,Charlie Brown,Engineering
bob jones,evan wright,Marketing


In [15]:
%%sql
CREATE TABLE departments (
    dept_name TEXT,
    manager TEXT
);
INSERT INTO departments VALUES ('Engineering', 'Sarah Connor');
INSERT INTO departments VALUES ('Marketing', 'Michael Scott');
INSERT INTO departments VALUES ('Finance', 'Bruce Wayne'); -- Note: No employees are in Finance yet


 * sqlite://
Done.
1 rows affected.
1 rows affected.
1 rows affected.


[]

In [17]:
#10.
# INNER JOIN - Returns only the rows where there is a match in both tables. (Diana Prince is excluded because 'HR' is not in the departments table; 'Finance' is excluded because it has no employees).
# FROM employees e INNER JOIN departments d: Opens both the employees (nicknamed e) and departments (nicknamed d) tables simultaneously.
# ON e.department = d.dept_name: The bridge that connects them. It matches the department column in the employee records with the department name in the department records.
# SELECT e.name, e.department, d.manager: Displays the final output—pulling the worker's name and department from the employee table, and their specific boss's name from the department table.

%%sql
SELECT e.name, e.department, d.manager
FROM employees e
INNER JOIN departments d
    ON e.department = d.dept_name;


 * sqlite://
Done.


name,department,manager
alice smith,Engineering,Sarah Connor
bob jones,Marketing,Michael Scott
Charlie Brown,Engineering,Sarah Connor
evan wright,Marketing,Michael Scott


In [18]:
# 11.
# LEFT JOIN (Left Outer Join) - Returns all employees, plus their manager if a match exists. If no match exists (like Diana in HR), it returns NULL.
# LEFT JOIN: Prioritizes the left table (employees). It guarantees that every single employee will appear in the final results.
# ON e.department = d.dept_name: Looks for a matching department to pull the manager's name.

%%sql
SELECT e.name, e.department, d.manager
FROM employees e
LEFT JOIN departments d
    ON e.department = d.dept_name;


 * sqlite://
Done.


name,department,manager
alice smith,Engineering,Sarah Connor
bob jones,Marketing,Michael Scott
Charlie Brown,Engineering,Sarah Connor
Diana prince,HR,None
evan wright,Marketing,Michael Scott


In [20]:
#12.
# RIGHT JOIN (Right Outer Join)- Returns all departments, even if they have no employees (like Finance). Employees in departments not listed in the departments table (like HR) are excluded.
# ON e.department = d.dept_name: Looks for matching employees who belong to each department.

%%sql
SELECT e.name, d.dept_name, d.manager
FROM employees e
RIGHT JOIN departments d
    ON e.department = d.dept_name;


 * sqlite://
Done.


name,dept_name,manager
alice smith,Engineering,Sarah Connor
bob jones,Marketing,Michael Scott
Charlie Brown,Engineering,Sarah Connor
evan wright,Marketing,Michael Scott
None,Finance,Bruce Wayne
